In [ ]:
"""
generator.py

Core synthetic QRIS merchant transaction data generator.

This module holds the real, importable logic — it's the source of truth.
Exploration/plotting/parameter-tuning happens in exploration.ipynb, which
imports from here rather than duplicating logic.

Start point: a single working function for the "steady" archetype
(e.g. warung sembako — consistent daily revenue, low variance).
Once this looks right when plotted, generalize into the fuller
parameterized generate_merchant() described in the design doc.
"""

import numpy as np
import pandas as pd


def generate_steady_merchant(
    days: int = 180,
    base_revenue: float = 500_000,
    noise_sigma: float = 0.3,
    weekend_bump: float = 1.15,
    start_date: str = "2025-01-01",
    seed: int | None = None,
) -> pd.DataFrame:
    """
    Generate a daily revenue series for a 'steady' archetype merchant.

    revenue(t) = base_revenue * weekend_bump(t) * lognormal_noise(t)

    Parameters
    ----------
    days : number of days to generate (brief recommends 6-12 months)
    base_revenue : the merchant's typical daily revenue level (IDR)
    noise_sigma : log-normal sigma controlling day-to-day variability.
        Higher = more erratic daily swings even though the archetype
        stays "steady" on average.
    weekend_bump : multiplier applied on Sat/Sun (location_type: "residential"
        default assumption from the design doc). Set to 1.0 for no weekly
        pattern, or <1.0 to simulate an office_district-style dip instead.
    start_date : first date of the generated series
    seed : optional RNG seed for reproducibility while tuning parameters

    Returns
    -------
    pd.DataFrame with columns: date, revenue
    """
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start_date, periods=days, freq="D")

    is_weekend = dates.weekday >= 5
    weekly_multiplier = np.where(is_weekend, weekend_bump, 1.0)

    noise = rng.lognormal(mean=0.0, sigma=noise_sigma, size=days)

    revenue = base_revenue * weekly_multiplier * noise

    return pd.DataFrame({"date": dates, "revenue": revenue})


if __name__ == "__main__":
    # Quick smoke test when running this file directly:
    #   python generator.py
    df = generate_steady_merchant(seed=42)
    print(df.head())
    print(f"\nGenerated {len(df)} days, mean revenue: {df['revenue'].mean():,.0f}")

In [4]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(42)  # reproducibility while you're iterating

In [5]:
def get_weekly_multiplier(dates, location_type="residential"):
    """
    Returns a per-day multiplier array based on location_type,
    per the design doc's weekly seasonality modifiers.
    """
    is_weekend = dates.weekday >= 5

    if location_type == "residential":
        # mild weekend bump
        return np.where(is_weekend, 1.15, 1.0)

    elif location_type == "office_district":
        # weekday-high, sharp weekend drop
        return np.where(is_weekend, 0.4, 1.0)

    elif location_type == "attraction":
        # weekend-high, weekday-low
        return np.where(is_weekend, 1.7, 0.85)

    elif location_type == "market":
        # roughly flat by day-of-week
        return np.ones(len(dates))

    else:
        raise ValueError(f"Unknown location_type: {location_type}")

In [6]:
def generate_steady_merchant(days=180, base_revenue=500_000, noise_sigma=0.08, location_type="residential"):
    dates = pd.date_range("2025-01-01", periods=days, freq="D")
    revenue = []
    for d in dates:
        weekly_multiplier = get_weekly_multiplier(d, location_type)
        daily = base_revenue * weekly_multiplier * np.random.lognormal(mean=0, sigma=noise_sigma)
        revenue.append(daily)
    return pd.DataFrame({"date": dates, "revenue": revenue})

df_warung = generate_steady_merchant(location_type = "residential")
df_warung.plot(x="date", y="revenue", figsize=(12,4))
plt.show()

TypeError: '>=' not supported between instances of 'method' and 'int'